# Lab07.2 — AXI DMA Image Threshold Accelerator

**Objective:** load a custom Verilog accelerator into PL, transfer a complete image from DDR to FPGA and back using **AXI DMA**, and compare the resulting binary mask **pixel-by-pixel** against NumPy.

**Prerequisite:** use the matching `lab07.bit` and `lab07.hwh` files in `Lab07/overlays/`. This notebook does **not** simulate FPGA output if the hardware is absent.

### Step 1 — Prepare the image and reference mask

Import the reference function and load the same grayscale scene as 07.1. Print its dimensions and the expected white-pixel count.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('../tools').resolve()))
from dma_threshold import ThresholdDMA, reference_threshold, locate_overlay

input_image = cv2.imread('../images/scene_256.png', cv2.IMREAD_GRAYSCALE)
if input_image is None: raise FileNotFoundError('Missing Lab07/images/scene_256.png')
threshold = 128
reference = reference_threshold(input_image, threshold)
print('Image:', input_image.shape, 'dtype:', input_image.dtype, 'bytes:', input_image.nbytes)
print('Expected foreground pixels:', np.count_nonzero(reference))

### Step 2 — Load the hardware overlay and inspect its IP

Check that both files are present, then program the FPGA. The overlay should contain an AXI DMA engine, AXI GPIO for the threshold, and the custom RTL streaming module. **This step replaces the currently loaded PL overlay**.

In [ ]:
bit_path, hwh_path = locate_overlay()
print('Bitstream:', bit_path)
print('HWH:', hwh_path)
fpga = ThresholdDMA()
for name, info in sorted(fpga.overlay.ip_dict.items()):
    print('{:<26} {}'.format(name, info.get('type', 'module')))
assert 'axi_dma_0' in fpga.overlay.ip_dict
assert 'threshold_gpio' in fpga.overlay.ip_dict

### Step 3 — Inspect FPGA control and transfer capabilities

The `threshold_gpio` DATA register at offset `0x00` sets a runtime-adjustable 8-bit threshold; TRI `0x04` sets it as an output. DMA operates in **simple mode** with **MM2S** (memory → stream) and **S2MM** (stream → memory).

In [ ]:
print('Current threshold register:', fpga._gpio.read(0x00) & 0xFF)
print('Threshold GPIO TRI register:', hex(fpga._gpio.read(0x04)))
print('DMA send running:', fpga.dma.sendchannel.running)
print('DMA receive running:', fpga.dma.recvchannel.running)
print('Maximum DMA buffer size (bytes):', fpga.dma.buffer_max_size)

### Step 4 — Process one image on FPGA

`process()` allocates physically contiguous buffers, packs the pixels into 32-bit words, starts the **receive channel first**, sends the image to PL, waits for DMA completion and copies the returned mask. The reported `dma_us` is **host-observed DMA transaction time**, not the internal propagation latency of one threshold comparator.

In [ ]:
fpga_mask, times = fpga.process(input_image, threshold)
print('Timings:', {k: round(v, 2) for k, v in times.items()})
print('Returned image:', fpga_mask.shape, fpga_mask.dtype)
print('Output values:', np.unique(fpga_mask))
assert fpga_mask.shape == input_image.shape
assert np.array_equal(fpga_mask, reference), 'FPGA differs from the reference algorithm!'
print('PASS — bit-exact match on all {} pixels'.format(input_image.size))

### Step 5 — Visual checkpoint: original, software, hardware and differences

Any disagreement appears in the fourth panel. A fully black difference image indicates **zero mismatched pixels**. The first three panels should have the same spatial structure as in 07.1.

In [ ]:
difference = cv2.absdiff(fpga_mask, reference)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, picture, title in zip(axes,
       (input_image, reference, fpga_mask, difference),
       ('Input grayscale', 'NumPy mask', 'FPGA mask', 'Difference')):
    ax.imshow(picture, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title); ax.axis('off')
plt.show()
print('Mismatched pixels:', np.count_nonzero(difference))

### Step 6 — Exercise the threshold register with different values

Changing only the threshold register changes the FPGA behavior: the bitstream remains the same. For each threshold, validate every output pixel against NumPy before displaying results.

In [ ]:
levels = (0, 64, 128, 192, 255)
fig, axes = plt.subplots(1, len(levels), figsize=(16, 3.5))
for ax, level in zip(axes, levels):
    hardware, timing = fpga.process(input_image, level)
    software = reference_threshold(input_image, level)
    assert np.array_equal(hardware, software), 'Mismatch at threshold {}'.format(level)
    white = np.count_nonzero(hardware) / hardware.size * 100
    print('threshold {:3d} | white {:5.1f}% | total {:.1f} us'.format(level, white, timing['total_us']))
    ax.imshow(hardware, cmap='gray', vmin=0, vmax=255)
    ax.set_title('T = {}\nwhite {:.1f}%'.format(level, white))
    ax.axis('off')
plt.show()

### Step 7 — Boundary test on every possible 8-bit pixel value

Generate a 16×16 image containing pixel values 0 to 255. This is an exhaustive test of the **threshold comparison function** at multiple thresholds; it is not an exhaustive test of DMA timing.

In [ ]:
all_values = np.arange(256, dtype=np.uint8).reshape(16,16)
for level in (0,1,127,128,129,254,255):
    hardware, timing = fpga.process(all_values, level)
    expected = reference_threshold(all_values, level)
    assert np.array_equal(hardware, expected), 'Mismatch at threshold {}'.format(level)
print('PASS — all 256 intensity levels tested at seven thresholds')

## Student exercises

1. What is the role of the PS high-performance AXI port HP0?
2. Why is the output receive DMA programmed before starting the input transfer?
3. What would happen to the receiving DMA if the RTL never asserted `TLAST`?
4. Compare the timing numbers and describe why `total_us` exceeds or equals `dma_us`.
5. Explain why the video/image array width is constrained to a multiple of four in this introductory design.